# Calibration for Binary Classification
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/google/yggdrasil-decision-forests/blob/main/documentation/public/docs/tutorial/binary_calibration.ipynb)


## Setup

In [1]:
pip install ydf -U

## Why Calibrate?

After training a binary classification model (e.g., predicting whether an email is `SPAM` or `NOT_SPAM`), the predicted probabilities are often not representative of the actual observed rates of the positive label.
For example, a binary classifier might output a predicted probability of 30%, but the true rate of observing the positive label for that prediction group could be significantly higher or lower than 30%.
Correcting this mismatch is conceptually straightforward, but it introduces a couple of key requirements that demand careful consideration:

* **Prediction Ordering**: Whatever method is used to adjust the predicted probabilities may or may not preserve the relative ordering of predictions.
Most popular calibration techniques (such as temperature scaling for logits or isotonic regression for probabilities) do preserve this ordering.
YDF's calibration methodology is based on isotonic regression and, therefore, preserves the relative ordering of predictions.
* **Threshold Preservation**: Determining final class assignments requires applying a decision threshold to the model's continuous output.
A binary classifier typically defaults to a threshold of `0.0` for logits (or `0.5` for probabilities).
Adjusting a model's predictions during calibration can shift certain values across this threshold, altering the model's overall quality metrics (such as precision and recall).
YDF's calibration methodology may shift predictions across a threshold, meaning you may need to tune the decision threshold after calibration to recover your pre-calibrated quality metrics.

## Binary classification model

For calibration, let's start by training a binary classification model on a real dataset.
We'll use the classic "Adult" dataset for this tutorial.

In [2]:
# Load necessary libraries
import ydf          # Yggdrasil Decision Forests.
import pandas as pd # To load and inspect the dataset.

# Download a classification dataset and load it as a Pandas DataFrame.
ds_path = "https://raw.githubusercontent.com/google/yggdrasil-decision-forests/main/yggdrasil_decision_forests/test_data/dataset"
train_ds = pd.read_csv(f"{ds_path}/adult_train.csv")
test_ds = pd.read_csv(f"{ds_path}/adult_test.csv")

The data is already partitioned into train and test splits.
However, when calibrating a model it is advised that you reserve an additional held-out portion of data specifically for this purpose.

In [3]:
# We will reserve 10% of the training data for calibration.
cal_ds = train_ds.sample(frac = 0.10, random_state=0)
train_ds = train_ds.drop(cal_ds.index)

The data for each split should not have any systemic differences.
Here, we'll examine the label rates for each split which should be virtually identical for calibration to work properly.

In [4]:
train_label_rates = train_ds.income.value_counts(normalize=True)
test_label_rates = test_ds.income.value_counts(normalize=True)
cal_label_rates = cal_ds.income.value_counts(normalize=True)

train_label_rates = train_label_rates.rename('train')
test_label_rates = test_label_rates.rename('test')
cal_label_rates = cal_label_rates.rename('cal')

pd.concat([train_label_rates, test_label_rates, cal_label_rates], axis=1)

,train,test,cal
income,,,
<=50K,0.75986,0.758727,0.755156
>50K,0.24014,0.241273,0.244844


Rates for each split are roughly equal, so at least along this axis of concern the calibration has a good chance of being effective.

The task is binary classification: predict whether an individual's income is `>50k` or `<=50k` based on other numerical and categorical features.
This dataset also contains missing values, which YDF handles automatically.

In [5]:
# Note: task=ydf.Task.CLASSIFICATION is the default, so it's optional here.
model = ydf.RandomForestLearner(label="income").train(train_ds)

Train model on 20513 examples
Model trained in 0:00:01.983004


The evaluation plots for a binary classification model include a [reliability diagram](https://scikit-learn.org/stable/modules/calibration.html#calibration-curves).
This diagram reveals how well the predicted probabilities reflect the frequency of the true label.
Ideally, this curve is a $45^\circ$ line between $(0,0)$ and $(1,1)$.
The error band around the curve is proportional to the amount of data used to determine the reliability at that point (specifically, a Beta-binomial error model).

Two calibration-related metrics are also included: the MCE and ECE.
These two metrics summarize how well the actual curve matches the ideal.

In [6]:
evaluation = model.evaluate(test_ds)
evaluation

Label \ Pred,<=50K,>50K
<=50K,6964,448
>50K,888,1469


As is clear from the reliability diagram, the model is poorly calibrated.
The curve is significantly different from the ideal.

Let's run YDF's calibration routine to modify the model's prediction such that they better match the observed frequencies in the data.

## Calibrate

A model has a `calibrate` method that accepts a dataset who's labels the model's predictions are calibrated.
The method also requires a `ydf.CalibrationConfig`.

Here we'll calibrate on specially reserved dataset that is 10% of the original dataset.

In [7]:
# Calibrate the model in place.
model.calibrate(
    cal_ds,
    ydf.CalibrationConfig(
        type=ydf.CalibrationType.SMOOTHED_PAV,
        n_bins=10000,
        z_threshold=0.5,
        n_grid=20000
    ),
)

Let's re-evaluate on the test set to see if the reliability diagram has improved.

In [9]:
evaluation = model.evaluate(test_ds)
evaluation

Label \ Pred,<=50K,>50K
<=50K,6964,448
>50K,888,1469


Both the metrics and reliability diagram show substantial improvements while leaving the quality metrics unchanged.
The calibration hyperparameters are detailed [here](https://ydf.readthedocs.io/en/latest/tutorial/pav_calibration_theory/) and you may need to tune them to your use case.

### Technical Note

Normal isotonic regression fitting for calibration will perfectly align the data used for calibration with the ideal reliability diagram (i.e. the $45^\circ$ diagonal).
However, the method used in YDF uses isotonic regression as a starting point for further smoothing and interpolation.
Thus, the performance of the calibration on the data used for calibration will _not_ be perfectly calibrated.
See [here](https://ydf.readthedocs.io/en/latest/tutorial/pav_calibration_theory/) for a detailed explanation for why this behavior is expected.

In [10]:
evaluation = model.evaluate(cal_ds)
evaluation

Label \ Pred,<=50K,>50K
<=50K,1607,114
>50K,215,343


Notice that the reliability diagram is near the ideal, but clearly diverges especially in the low-population bins.
Performance on other datasets may have an even larger divergence.
Again, tune the calibration parameters until you attain a reliability diagram and set of metrics suitable for your application.